# 01 Data Joining — Men's

This notebook joins all men's raw data sources into clean, unified datasets for downstream EDA, feature engineering, and modeling.

**Inputs** (from `00_data_collection/`):
- `MRegularSeasonCompactResults.csv` — game scores, 1985–2026
- `MRegularSeasonDetailedResults.csv` — box scores, 2003–2026
- `MNCAATourneyCompactResults.csv` — tournament scores, 1985–2025
- `MNCAATourneyDetailedResults.csv` — tournament box scores, 2003–2025
- `MNCAATourneySeeds.csv` — tournament seeds
- `MMasseyOrdinals.csv` — weekly rankings from ~197 systems, 2003–2026
- `MTeams.csv`, `MTeamConferences.csv`, `Conferences.csv`, `MSeasons.csv`, `MTeamCoaches.csv`

**Outputs** (to S3 `01_data_joining/mens/`):
1. `regular_season_games.parquet` — all regular season games with detailed stats where available
2. `tourney_games.parquet` — all tournament games with detailed stats where available
3. `team_season_stats.parquet` — per-team per-season aggregated statistics
4. `tourney_seeds.parquet` — cleaned tournament seeds with numeric seed extracted
5. `massey_ordinals_pre_tourney.parquet` — pre-tournament Massey rankings (DayNum < 132)
6. `team_metadata.parquet` — team info, conferences, coaches per season

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

#### Functions

In [2]:
def read_csv(filename):
    """Read CSV from S3 if available, otherwise fall back to local."""
    try:
        return pd.read_csv(f"{INPUT_PREFIX}{filename}")
    except Exception:
        return pd.read_csv(f"{LOCAL_DATA}{filename}")

In [3]:
def aggregate_team_season(team_games_df):
    """
    Compute per-team per-season aggregate stats from team-centric game rows.
    """
    box_cols = ['FGM', 'FGA', 'FGM3', 'FGA3', 'FTM', 'FTA', 'OR', 'DR', 'Ast', 'TO', 'Stl', 'Blk', 'PF']
    opp_box_cols = [f'Opp{c}' for c in box_cols]
    
    agg_dict = {
        'Win': ['sum', 'count'],
        'Score': 'mean',
        'OppScore': 'mean',
    }
    
    # Add box score aggregations if available
    for col in box_cols + opp_box_cols:
        if col in team_games_df.columns:
            agg_dict[col] = 'mean'
    
    stats = team_games_df.groupby(['Season', 'TeamID']).agg(agg_dict)
    
    # Flatten MultiIndex columns
    stats.columns = ['_'.join(col).strip('_') for col in stats.columns]
    stats = stats.rename(columns={
        'Win_sum': 'Wins',
        'Win_count': 'Games',
        'Score_mean': 'AvgScore',
        'OppScore_mean': 'AvgOppScore',
    })
    
    # Rename box score columns to Avg prefix
    for col in box_cols + opp_box_cols:
        if f'{col}_mean' in stats.columns:
            stats = stats.rename(columns={f'{col}_mean': f'Avg{col}'})
    
    stats['Losses'] = stats['Games'] - stats['Wins']
    stats['WinPct'] = stats['Wins'] / stats['Games']
    stats['AvgPointDiff'] = stats['AvgScore'] - stats['AvgOppScore']
    
    # Derived shooting percentages
    if 'AvgFGM' in stats.columns:
        stats['FGPct'] = stats['AvgFGM'] / stats['AvgFGA']
        stats['FG3Pct'] = stats['AvgFGM3'] / stats['AvgFGA3']
        stats['FTPct'] = stats['AvgFTM'] / stats['AvgFTA']
        stats['OppFGPct'] = stats['AvgOppFGM'] / stats['AvgOppFGA']
        stats['OppFG3Pct'] = stats['AvgOppFGM3'] / stats['AvgOppFGA3']
        
        # Possessions estimate (per game average)
        # Possessions ≈ FGA - OR + TO + 0.475 * FTA
        stats['AvgPoss'] = (stats['AvgFGA'] - stats['AvgOR'] + stats['AvgTO'] + 0.475 * stats['AvgFTA'])
        stats['AvgOppPoss'] = (stats['AvgOppFGA'] - stats['AvgOppOR'] + stats['AvgOppTO'] + 0.475 * stats['AvgOppFTA'])
        
        # Offensive and Defensive Efficiency (points per possession)
        stats['OffEff'] = stats['AvgScore'] / stats['AvgPoss']
        stats['DefEff'] = stats['AvgOppScore'] / stats['AvgOppPoss']
        stats['NetEff'] = stats['OffEff'] - stats['DefEff']
    
    stats = stats.reset_index()
    return stats

#### Constants

In [4]:
BUCKET = "march-machine-learning-mania-2026"
GENDER = "mens"
STAGE = "01_data_joining"
INPUT_PREFIX = f"s3://{BUCKET}/00_data_collection/"
OUTPUT_PREFIX = f"s3://{BUCKET}/{STAGE}/{GENDER}/"

# For local development, fall back to local data directory
LOCAL_DATA = "../00_data_collection/"
LOCAL_OUTPUT = "output/"

#### Make output directory

In [5]:
os.makedirs(LOCAL_OUTPUT, exist_ok=True)

## 1. Load Raw Data

In [6]:
# Game results
reg_compact = read_csv("MRegularSeasonCompactResults.csv")
reg_detailed = read_csv("MRegularSeasonDetailedResults.csv")
tourney_compact = read_csv("MNCAATourneyCompactResults.csv")
tourney_detailed = read_csv("MNCAATourneyDetailedResults.csv")

# Tournament metadata
seeds = read_csv("MNCAATourneySeeds.csv")
seasons = read_csv("MSeasons.csv")

# Rankings
massey = read_csv("MMasseyOrdinals.csv")

# Team metadata
teams = read_csv("MTeams.csv")
team_conferences = read_csv("MTeamConferences.csv")
conferences = read_csv("Conferences.csv")
coaches = read_csv("MTeamCoaches.csv")

print(f"Regular season compact: {reg_compact.shape}")
print(f"Regular season detailed: {reg_detailed.shape}")
print(f"Tourney compact: {tourney_compact.shape}")
print(f"Tourney detailed: {tourney_detailed.shape}")
print(f"Seeds: {seeds.shape}")
print(f"Massey ordinals: {massey.shape}")
print(f"Teams: {teams.shape}")

Regular season compact: (198079, 8)
Regular season detailed: (124031, 34)
Tourney compact: (2585, 8)
Tourney detailed: (1449, 34)
Seeds: (2626, 3)
Massey ordinals: (5819228, 5)
Teams: (381, 4)


## 2. Join Regular Season Games

Merge compact and detailed results. Compact results cover 1985–2026, detailed cover 2003–2026. The merge preserves all compact rows, adding detailed stats as NaN for pre-2003 seasons.

In [7]:
# The detailed results contain all compact columns plus box score stats.
# We merge on the common columns to get one unified dataset.
compact_cols = reg_compact.columns.tolist()
detail_only_cols = [c for c in reg_detailed.columns if c not in compact_cols]

reg_games = reg_compact.merge(
    reg_detailed[compact_cols + detail_only_cols],
    on=compact_cols,
    how='left'
)

print(f"Regular season games: {reg_games.shape}")
print(f"Seasons with detailed stats: {reg_games.dropna(subset=['WFGM']).Season.nunique()}")
print(f"Seasons without detailed stats: {reg_games[reg_games['WFGM'].isna()].Season.nunique()}")

Regular season games: (198079, 34)
Seasons with detailed stats: 24
Seasons without detailed stats: 18


## 3. Join Tournament Games

Same approach for tournament results.

In [8]:
tourney_games = tourney_compact.merge(
    tourney_detailed[[c for c in tourney_detailed.columns]],
    on=compact_cols,
    how='left'
)

print(f"Tournament games: {tourney_games.shape}")
print(f"Tournament seasons: {tourney_games.Season.min()} - {tourney_games.Season.max()}")

Tournament games: (2585, 34)
Tournament seasons: 1985 - 2025


## 4. Build Team-Season Aggregated Statistics

For each team and season, compute aggregated regular season stats. We "unpivot" from winner/loser format to a team-centric view first, then aggregate.

This produces per-team-season averages that are the foundation for matchup features later.

In [9]:
def build_team_game_rows(games_df):
    """
    Convert winner/loser format into team-centric rows.
    Each game produces two rows: one for each team.
    """
    # Columns that exist in detailed results (box score stats)
    box_cols = ['FGM', 'FGA', 'FGM3', 'FGA3', 'FTM', 'FTA', 'OR', 'DR', 'Ast', 'TO', 'Stl', 'Blk', 'PF']
    has_detail = f'WFGM' in games_df.columns and games_df['WFGM'].notna().any()
    
    rows = []
    
    # Winner rows
    w = games_df[['Season', 'DayNum', 'WTeamID', 'WScore', 'LTeamID', 'LScore', 'WLoc', 'NumOT']].copy()
    w.columns = ['Season', 'DayNum', 'TeamID', 'Score', 'OppID', 'OppScore', 'WLoc', 'NumOT']
    w['Win'] = 1
    # Map location: W=winner home -> team is H; L=winner away -> team is A; N=neutral
    w['Loc'] = w['WLoc'].map({'H': 'H', 'A': 'A', 'N': 'N'})
    
    # Loser rows
    l = games_df[['Season', 'DayNum', 'LTeamID', 'LScore', 'WTeamID', 'WScore', 'WLoc', 'NumOT']].copy()
    l.columns = ['Season', 'DayNum', 'TeamID', 'Score', 'OppID', 'OppScore', 'WLoc', 'NumOT']
    l['Win'] = 0
    # Loser location is opposite of winner's
    l['Loc'] = l['WLoc'].map({'H': 'A', 'A': 'H', 'N': 'N'})
    
    if has_detail:
        for col in box_cols:
            w[col] = games_df[f'W{col}'].values
            w[f'Opp{col}'] = games_df[f'L{col}'].values
            l[col] = games_df[f'L{col}'].values
            l[f'Opp{col}'] = games_df[f'W{col}'].values
    
    w = w.drop(columns=['WLoc'])
    l = l.drop(columns=['WLoc'])
    
    return pd.concat([w, l], ignore_index=True)

team_games = build_team_game_rows(reg_games)
print(f"Team-game rows: {team_games.shape}")
team_games.head()

Team-game rows: (396158, 35)


,Season,DayNum,TeamID,Score,OppID,OppScore,NumOT,Win,Loc,FGM,...,Ast,OppAst,TO,OppTO,Stl,OppStl,Blk,OppBlk,PF,OppPF
0,1985,20,1228,81,1328,64,0,1,N,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1985,25,1106,77,1354,70,0,1,H,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1985,25,1112,63,1223,56,0,1,H,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1985,25,1165,70,1432,54,0,1,H,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1985,25,1192,86,1447,74,0,1,H,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
team_season_stats = aggregate_team_season(team_games)
print(f"Team-season stats: {team_season_stats.shape}")
print(f"Columns: {team_season_stats.columns.tolist()}")
team_season_stats.head()

Team-season stats: (13753, 45)
Columns: ['Season', 'TeamID', 'Wins', 'Games', 'AvgScore', 'AvgOppScore', 'AvgFGM', 'AvgFGA', 'AvgFGM3', 'AvgFGA3', 'AvgFTM', 'AvgFTA', 'AvgOR', 'AvgDR', 'AvgAst', 'AvgTO', 'AvgStl', 'AvgBlk', 'AvgPF', 'AvgOppFGM', 'AvgOppFGA', 'AvgOppFGM3', 'AvgOppFGA3', 'AvgOppFTM', 'AvgOppFTA', 'AvgOppOR', 'AvgOppDR', 'AvgOppAst', 'AvgOppTO', 'AvgOppStl', 'AvgOppBlk', 'AvgOppPF', 'Losses', 'WinPct', 'AvgPointDiff', 'FGPct', 'FG3Pct', 'FTPct', 'OppFGPct', 'OppFG3Pct', 'AvgPoss', 'AvgOppPoss', 'OffEff', 'DefEff', 'NetEff']


,Season,TeamID,Wins,Games,AvgScore,AvgOppScore,AvgFGM,AvgFGA,AvgFGM3,AvgFGA3,...,FGPct,FG3Pct,FTPct,OppFGPct,OppFG3Pct,AvgPoss,AvgOppPoss,OffEff,DefEff,NetEff
0,1985,1102,5,24,63.083333,68.875000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1985,1103,9,23,61.043478,64.086957,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1985,1104,21,30,68.500000,60.700000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1985,1106,10,24,71.625000,75.416667,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1985,1108,19,25,83.000000,75.040000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 5. Clean Tournament Seeds

Extract the numeric seed from the seed string (e.g., `W01` → 1, `X16a` → 16). The region letter and play-in indicator are stored separately.

In [11]:
tourney_seeds = seeds.copy()
tourney_seeds['Region'] = tourney_seeds['Seed'].str[0]
tourney_seeds['SeedNum'] = tourney_seeds['Seed'].str[1:3].astype(int)
tourney_seeds['PlayIn'] = tourney_seeds['Seed'].str[3:]  # 'a', 'b', or ''

print(f"Tournament seeds: {tourney_seeds.shape}")
print(f"Seed range: {tourney_seeds.SeedNum.min()} - {tourney_seeds.SeedNum.max()}")
print(f"Seasons: {tourney_seeds.Season.min()} - {tourney_seeds.Season.max()}")
tourney_seeds.head()

Tournament seeds: (2626, 6)
Seed range: 1 - 16
Seasons: 1985 - 2025


,Season,Seed,TeamID,Region,SeedNum,PlayIn
0,1985,W01,1207,W,1,
1,1985,W02,1210,W,2,
2,1985,W03,1228,W,3,
3,1985,W04,1260,W,4,
4,1985,W05,1374,W,5,


## 6. Filter Massey Ordinals (Pre-Tournament Only)

Massey Ordinals include post-tournament rankings which would be data leakage. We filter to DayNum < 132 (before Selection Sunday) and keep only the latest available ranking per system per team per season.

In [12]:
# Filter to pre-tournament rankings only
massey_pre = massey[massey['RankingDayNum'] < 132].copy()
print(f"Massey rows total: {len(massey):,}")
print(f"Massey rows pre-tournament: {len(massey_pre):,}")
print(f"Unique systems: {massey_pre.SystemName.nunique()}")
print(f"Seasons: {massey_pre.Season.min()} - {massey_pre.Season.max()}")

Massey rows total: 5,819,228
Massey rows pre-tournament: 5,424,074
Unique systems: 191
Seasons: 2003 - 2026


In [13]:
# Keep only the latest pre-tournament ranking per system/team/season
massey_latest = (
    massey_pre
    .sort_values('RankingDayNum')
    .groupby(['Season', 'SystemName', 'TeamID'])
    .last()
    .reset_index()
)

print(f"Latest pre-tournament rankings: {massey_latest.shape}")
massey_latest.head()

Latest pre-tournament rankings: (447057, 5)


,Season,SystemName,TeamID,RankingDayNum,OrdinalRank
0,2003,AP,1104,93,22
1,2003,AP,1112,127,1
2,2003,AP,1120,79,24
3,2003,AP,1143,127,24
4,2003,AP,1158,37,25


In [14]:
# Pivot to wide format: one row per (Season, TeamID), one column per system
massey_wide = massey_latest.pivot_table(
    index=['Season', 'TeamID'],
    columns='SystemName',
    values='OrdinalRank'
).reset_index()

# Flatten column names
massey_wide.columns.name = None

# Compute average rank across all available systems per team-season
system_cols = [c for c in massey_wide.columns if c not in ['Season', 'TeamID']]
massey_wide['AvgOrdinalRank'] = massey_wide[system_cols].mean(axis=1)

# Also compute average of the historically best systems identified in research:
# POM (Pomeroy), SAG (Sagarin), MOR (Moore), WLK (Whitlock)
top_systems = ['POM', 'SAG', 'MOR', 'WLK']
available_top = [s for s in top_systems if s in massey_wide.columns]
if available_top:
    massey_wide['TopSystemsAvgRank'] = massey_wide[available_top].mean(axis=1)
    print(f"Top systems available: {available_top}")

print(f"Massey wide format: {massey_wide.shape}")
print(f"Systems as columns: {len(system_cols)}")
massey_wide.head()

Top systems available: ['POM', 'SAG', 'MOR', 'WLK']
Massey wide format: (8356, 195)
Systems as columns: 191


,Season,TeamID,7OT,ACU,ADE,AEI,AP,ARG,ATP,AUS,...,WLS,WMR,WMV,WOB,WOL,WTE,YAG,ZAM,AvgOrdinalRank,TopSystemsAvgRank
0,2003,1102,NaN,NaN,NaN,NaN,NaN,145.0,NaN,NaN,...,NaN,NaN,NaN,153.0,158.0,147.0,NaN,NaN,155.617647,153.25
1,2003,1103,NaN,NaN,NaN,NaN,NaN,171.0,NaN,NaN,...,NaN,NaN,NaN,177.0,165.0,152.0,NaN,NaN,165.617647,157.50
2,2003,1104,NaN,NaN,NaN,NaN,22.0,34.0,NaN,NaN,...,NaN,NaN,NaN,32.0,33.0,28.0,NaN,NaN,30.416667,27.00
3,2003,1105,NaN,NaN,NaN,NaN,NaN,309.0,NaN,NaN,...,NaN,NaN,NaN,312.0,312.0,304.0,NaN,NaN,309.176471,310.00
4,2003,1106,NaN,NaN,NaN,NaN,NaN,242.0,NaN,NaN,...,NaN,NaN,NaN,255.0,268.0,247.0,NaN,NaN,254.147059,263.75


## 7. Build Team Metadata

Join team names, conference affiliations, and coach information per season.

In [15]:
# Join conferences with their descriptions
team_conf = team_conferences.merge(conferences, on='ConfAbbrev', how='left')
team_conf = team_conf.rename(columns={'Description': 'Conference'})

# Identify power conferences (Power 6 + Big East historically)
power_conf_abbrevs = ['acc', 'big_ten', 'big_twelve', 'sec', 'pac_twelve', 'big_east']
team_conf['IsPowerConf'] = team_conf['ConfAbbrev'].isin(power_conf_abbrevs).astype(int)

# Join team names
team_meta = team_conf.merge(
    teams[['TeamID', 'TeamName']],
    on='TeamID',
    how='left'
)

# Join coach for each team-season (use the coach active at end of regular season)
# Filter coaches to those active through at least DayNum 132 (Selection Sunday)
coaches_latest = coaches[coaches['LastDayNum'] >= 132].copy()
coaches_latest = coaches_latest.drop_duplicates(subset=['Season', 'TeamID'], keep='last')

team_meta = team_meta.merge(
    coaches_latest[['Season', 'TeamID', 'CoachName']],
    on=['Season', 'TeamID'],
    how='left'
)

print(f"Team metadata: {team_meta.shape}")
team_meta.head()

Team metadata: (13753, 7)


,Season,TeamID,ConfAbbrev,Conference,IsPowerConf,TeamName,CoachName
0,1985,1102,wac,Western Athletic Conference,0,Air Force,reggie_minton
1,1985,1103,ovc,Ohio Valley Conference,0,Akron,bob_huggins
2,1985,1104,sec,Southeastern Conference,1,Alabama,wimp_sanderson
3,1985,1106,swac,Southwest Athletic Conference,0,Alabama St,james_oliver
4,1985,1108,swac,Southwest Athletic Conference,0,Alcorn St,davey_whitney


## 8. Save Outputs

Write all joined datasets to S3 (parquet format for efficiency).

In [16]:
outputs = {
    'regular_season_games': reg_games,
    'tourney_games': tourney_games,
    'team_season_stats': team_season_stats,
    'tourney_seeds': tourney_seeds,
    'massey_ordinals_pre_tourney': massey_wide,
    'team_metadata': team_meta,
}

for name, df in outputs.items():
    # Save to S3
    try:
        s3_path = f"{OUTPUT_PREFIX}{name}.parquet"
        df.to_parquet(s3_path, index=False)
        print(f"Saved to S3: {s3_path} ({df.shape})")
    except Exception as e:
        print(f"S3 save failed for {name}: {e}")

Saved to S3: s3://march-machine-learning-mania-2026/01_data_joining/mens/regular_season_games.parquet ((198079, 34))
Saved to S3: s3://march-machine-learning-mania-2026/01_data_joining/mens/tourney_games.parquet ((2585, 34))
Saved to S3: s3://march-machine-learning-mania-2026/01_data_joining/mens/team_season_stats.parquet ((13753, 45))
Saved to S3: s3://march-machine-learning-mania-2026/01_data_joining/mens/tourney_seeds.parquet ((2626, 6))
Saved to S3: s3://march-machine-learning-mania-2026/01_data_joining/mens/massey_ordinals_pre_tourney.parquet ((8356, 195))
Saved to S3: s3://march-machine-learning-mania-2026/01_data_joining/mens/team_metadata.parquet ((13753, 7))


## 9. Output Summary

Quick summary of what was produced for downstream stages.

In [17]:
print("=" * 60)
print("DATA JOINING SUMMARY — MEN'S")
print("=" * 60)
for name, df in outputs.items():
    print(f"\n{name}:")
    print(f"  Shape: {df.shape}")
    print(f"  Columns: {df.columns.tolist()[:10]}{'...' if len(df.columns) > 10 else ''}")
    if 'Season' in df.columns:
        print(f"  Seasons: {df.Season.min()} - {df.Season.max()}")

DATA JOINING SUMMARY — MEN'S

regular_season_games:
  Shape: (198079, 34)
  Columns: ['Season', 'DayNum', 'WTeamID', 'WScore', 'LTeamID', 'LScore', 'WLoc', 'NumOT', 'WFGM', 'WFGA']...
  Seasons: 1985 - 2026

tourney_games:
  Shape: (2585, 34)
  Columns: ['Season', 'DayNum', 'WTeamID', 'WScore', 'LTeamID', 'LScore', 'WLoc', 'NumOT', 'WFGM', 'WFGA']...
  Seasons: 1985 - 2025

team_season_stats:
  Shape: (13753, 45)
  Columns: ['Season', 'TeamID', 'Wins', 'Games', 'AvgScore', 'AvgOppScore', 'AvgFGM', 'AvgFGA', 'AvgFGM3', 'AvgFGA3']...
  Seasons: 1985 - 2026

tourney_seeds:
  Shape: (2626, 6)
  Columns: ['Season', 'Seed', 'TeamID', 'Region', 'SeedNum', 'PlayIn']
  Seasons: 1985 - 2025

massey_ordinals_pre_tourney:
  Shape: (8356, 195)
  Columns: ['Season', 'TeamID', '7OT', 'ACU', 'ADE', 'AEI', 'AP', 'ARG', 'ATP', 'AUS']...
  Seasons: 2003 - 2026

team_metadata:
  Shape: (13753, 7)
  Columns: ['Season', 'TeamID', 'ConfAbbrev', 'Conference', 'IsPowerConf', 'TeamName', 'CoachName']
  Seasons: